# 08.6 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/08-deep-learning/06-training-well/exercises.ipynb)

In [ ]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"        # hide TensorFlow start-up messages
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

import keras
from keras import layers
keras.utils.set_random_seed(42)      # same starting weights every run (small differences across machines are normal)
print("Keras", keras.__version__, "on", keras.backend.backend())

import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail
from sklearn.model_selection import train_test_split

raw = load_retail("Year 2010-2011")
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

cutoff = pd.Timestamp("2011-09-01")
past = sales[sales["InvoiceDate"] < cutoff]
future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < "2011-12-01")]
cust = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
                                       products=("StockCode", "nunique"),
                                       first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
cust["tenure_days"] = (cutoff - cust["first_order"]).dt.days
cust["days_since_last"] = (cutoff - cust["last_order"]).dt.days
future_spend = future.groupby("Customer ID")["Revenue"].sum().reindex(cust.index).fillna(0)
cust["churned"] = (future_spend == 0).astype(int)
cust = cust.drop(columns=["first_order", "last_order"])

features = ["orders", "spend", "products", "tenure_days", "days_since_last"]
X_train, X_test, y_train, y_test = train_test_split(
    cust[features], cust["churned"], test_size=0.25, random_state=42, stratify=cust["churned"])
print(f"{len(cust):,} customers, churn rate {cust['churned'].mean():.1%}, "
      f"train {len(X_train):,} / test {len(X_test):,}")

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(X_train)            # fit on training customers only (lesson 04.5)
Xs_train, Xs_test = scaler.transform(X_train), scaler.transform(X_test)

from sklearn.metrics import roc_auc_score
y_tr = y_train.to_numpy()
X_small, y_small = Xs_train[:300], y_tr[:300]        # a deliberately tiny training set
X_val, y_val = Xs_train[300:], y_tr[300:]            # validation: the other training customers

def small_net(learning_rate=0.001, optimizer=None):
    keras.utils.set_random_seed(42)
    m = keras.Sequential([keras.Input((5,)), layers.Dense(16, activation="relu"),
                          layers.Dense(1, activation="sigmoid")])
    m.compile(optimizer=optimizer or keras.optimizers.Adam(learning_rate), loss="binary_crossentropy")
    return m

def big_net(dropout=0.0, l2=0.0, batchnorm=False):
    """Two layers of 256 neurons: far too big for 300 customers, on purpose."""
    keras.utils.set_random_seed(42)
    reg = keras.regularizers.L2(l2) if l2 else None
    stack = [keras.Input((5,))]
    for _ in range(2):
        stack.append(layers.Dense(256, activation="relu", kernel_regularizer=reg))
        if batchnorm: stack.append(layers.BatchNormalization())
        if dropout: stack.append(layers.Dropout(dropout))
    m = keras.Sequential(stack + [layers.Dense(1, activation="sigmoid")])
    m.compile(optimizer=keras.optimizers.Adam(0.003), loss="binary_crossentropy")
    return m

def test_auc(m):
    return roc_auc_score(y_test, m.predict(Xs_test, verbose=0).ravel())

## Exercise 1: momentum, two more steps

By hand, continue the README table for steps 4 and 5. Then check in code.

In [ ]:
# your code here

## Exercise 2: two more learning rates

Train `small_net(3.0)` and `small_net(0.03)` for 30 epochs. Which rows of the README table do they resemble?

In [ ]:
# your code here

## Exercise 3: dropout plus early stopping

On the 300-customer setup, combine `big_net(dropout=0.5)` with early stopping (patience 10). Better than either alone?

In [ ]:
# your code here

## Exercise 4: load the checkpoint

Use `ModelCheckpoint` to save the best epoch of `big_net()` to `models/`, then load it with `keras.models.load_model` and compute its test AUC.

In [ ]:
# your code here